In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from os import listdir

import matplotlib.pyplot as plt
from pendulibrary.continuation import adaptive_cont
from pendulibrary.common_targetters import single_fixed
from pendulibrary.plotters import compare_fast
from pendulibrary.integrate import integrate_state
from pendulibrary.common import hamiltonian

int_tol = 1e-14

In [ ]:
fname = "DDsp-P6b"

data = np.load(f"../database/{fname}.npz")
x0s_in = data["x0s"]
periods_in = data["periods"]
tangents_in = data["tangents"]
eigs_in = data["eigs"]
hams_in = data["hamiltonians"]
Lr, Mr = data["params"]


vals = np.column_stack((x0s_in, periods_in))
th0s_abs = np.abs(x0s_in[:, :2])
ind_fix = np.argmin(np.std(th0s_abs, axis=0))
val_fix = np.pi if abs(th0s_abs[0, ind_fix] - np.pi) < th0s_abs[0, ind_fix] else 0

targ = single_fixed(ind_fix, val_fix, 2, Lr, Mr, int_tol)
func = targ.g_dg_stm

In [ ]:
Xm1 = targ.get_X(x0s_in[-2], periods_in[-2])
Xm0 = targ.get_X(x0s_in[-1], periods_in[-1])

# Xm1 = targ.get_X(x0s_in[20], periods_in[20])
# Xm0 = targ.get_X(x0s_in[21], periods_in[21])
_, dg, _ = func(Xm0)
svd = np.linalg.svd(dg)
sprev = np.linalg.norm(Xm1 - Xm0)
print(f"Last dist is {sprev:.3e}")

In [ ]:
cont_kwargs = dict(
    s0=sprev, s_min=1e-7, S=25.0, tol=1e-9, max_iter=11, target_iter=5, rate=1.05
)
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    Xm0, func, Xm0 - Xm1, **cont_kwargs, exact_tangent=False, pseudo=False
)

In [ ]:
fnames = [f.removesuffix(".npz") for f in listdir("../database") if "DDsp" in f]
x0s_new = np.array([targ.get_x0(X) for X in Xs])
periods_new = np.array([targ.get_period(X) for X in Xs])
hams_new = hamiltonian(x0s_new.T, Lr, Mr)
compare_fast(periods_new, hams_new, fnames)

In [ ]:
x0s_tot = np.vstack((x0s_in, x0s_new[1:]))
periods_tot = np.concat((periods_in, periods_new[1:]))
eigs_tot = np.vstack((eigs_in, eig_vals[1:]))
tans = np.column_stack((np.zeros_like(periods_new), tangents))
tans_tot = np.vstack((tangents_in, tans[1:]))
hams_tot = np.concat((hams_in, hams_new[1:]))

In [ ]:
eigs_tot.shape

In [ ]:
x0s_tot.shape

In [ ]:
np.savez_compressed(
    f"../database/{fname}",
    x0s=x0s_tot,
    periods=periods_tot,
    eigs=eigs_tot,
    tangents=tans_tot,
    hamiltonians=hams_tot,
    params=np.array([Lr, Mr]),
)

In [ ]:
plt.close("all")
ts, xs, fs = integrate_state(x0s_new[-1], periods_new[-1], Lr, Mr, 1e-14)
y = -np.cos(xs[0]) - Lr * np.cos(xs[1])
x = np.sin(xs[0]) + Lr * np.sin(xs[1])

# plt.plot(xs.T)
plt.plot(x, y)
plt.axis("equal")
plt.show()